# Modul ML - Multi Layer Perceptron
Dokumentasi praktikum: implementasi MLP untuk dataset MNIST sesuai modul.


In [ ]:
# 1. Load Dataset MNIST
from tensorflow.keras.datasets import mnist  # import mnist dataset from keras
from tensorflow.keras.utils import to_categorical  # helper to one-hot encode labels
import numpy as np  # numerical operations
import matplotlib.pyplot as plt  # plotting
import seaborn as sns  # nicer plots
(X_train, y_train), (X_test, y_test) = mnist.load_data()  # load data dari Keras
print('Shapes:', X_train.shape, y_train.shape, X_test.shape, y_test.shape)  # tampilkan shape data


In [ ]:
# 2. Menampilkan distribusi data Test
# tampilkan jumlah sampel per kelas pada data test
import pandas as pd  # import pandas untuk frekuensi
pd.Series(y_test).value_counts().sort_index().plot(kind='bar')  # plot distribusi label test
plt.title('Distribusi label - Test set')  # judul plot
plt.xlabel('Label')  # label x
plt.ylabel('Count')  # label y
plt.show()  # tampilkan plot


In [ ]:
# 3. Menampilkan sample data Train
# tampilkan beberapa contoh gambar dari data train
plt.figure(figsize=(8,8))  # atur ukuran figura
for i in range(9):  # tampilkan 9 sample
    plt.subplot(3,3,i+1)  # grid 3x3
    plt.imshow(X_train[i], cmap='gray')  # tampilkan gambar grayscale
    plt.title(f'True: {y_train[i]}')  # judul menampilkan label asli
    plt.axis('off')  # sembunyikan axis
plt.tight_layout()  # rapikan layout
plt.show()  # tampilkan semua gambar


In [ ]:
# 4. Pra-proses data
# ubah tipe, normalisasi, flatten, dan one-hot encode label
X_train = X_train.astype('float32') / 255.0  # normalisasi ke rentang [0,1]
X_test = X_test.astype('float32') / 255.0  # normalisasi test
nsamples, nx, ny = X_train.shape  # dapatkan ukuran gambar
X_train_flat = X_train.reshape((nsamples, nx*ny))  # flatten training images
X_test_flat = X_test.reshape((X_test.shape[0], nx*ny))  # flatten test images
y_train_cat = to_categorical(y_train, num_classes=10)  # one-hot encode labels train
y_test_cat = to_categorical(y_test, num_classes=10)  # one-hot encode labels test
print('After preprocessing shapes:', X_train_flat.shape, y_train_cat.shape, X_test_flat.shape, y_test_cat.shape)  # cek shape


In [ ]:
# 5. Membuat model dan train model
from tensorflow.keras.models import Sequential  # import Sequential API
from tensorflow.keras.layers import Dense, Dropout  # import layer yang dibutuhkan
model = Sequential([  # definisikan arsitektur MLP
    Dense(512, activation='relu', input_shape=(nx*ny,)),  # hidden layer 1
    Dropout(0.2),  # dropout untuk regularisasi
    Dense(256, activation='relu'),  # hidden layer 2
    Dropout(0.2),  # dropout
    Dense(10, activation='softmax')  # layer output dengan softmax
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])  # compile model
history = model.fit(X_train_flat, y_train_cat, epochs=10, batch_size=128, validation_split=0.1)  # latih model


In [ ]:
# 6. Menampilkan grafik history train
# plot loss dan akurasi training dan validation
plt.figure(figsize=(12,4))  # buat figure lebar
plt.subplot(1,2,1)  # subplot untuk loss
plt.plot(history.history['loss'], label='train loss')  # plot train loss
plt.plot(history.history['val_loss'], label='val loss')  # plot val loss
plt.title('Loss selama training')  # judul
plt.xlabel('Epoch')  # label x
plt.ylabel('Loss')  # label y
plt.legend()  # tampilkan legend
plt.subplot(1,2,2)  # subplot untuk akurasi
plt.plot(history.history['accuracy'], label='train acc')  # plot train acc
plt.plot(history.history['val_accuracy'], label='val acc')  # plot val acc
plt.title('Akurasi selama training')  # judul
plt.xlabel('Epoch')  # label x
plt.ylabel('Accuracy')  # label y
plt.legend()  # tampilkan legend
plt.show()  # tampilkan grafik


In [ ]:
# 7. Menampilkan classification report
from sklearn.metrics import classification_report, accuracy_score  # import metrics dari sklearn
y_pred_proba = model.predict(X_test_flat)  # prediksi probabilitas pada test
y_pred = np.argmax(y_pred_proba, axis=1)  # ambil label dengan probabilitas tertinggi
print('Test accuracy (sklearn):', accuracy_score(y_test, y_pred))  # cetak akurasi menggunakan sklearn
print('\nClassification Report:\n', classification_report(y_test, y_pred))  # tampilkan classification report


In [ ]:
# 8. Membuat confusion matrix
from sklearn.metrics import confusion_matrix  # import confusion matrix
cm = confusion_matrix(y_test, y_pred)  # hitung confusion matrix
plt.figure(figsize=(10,8))  # set figure size
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')  # tampilkan heatmap CM dengan annotations
plt.title('Confusion Matrix')  # judul
plt.xlabel('Predicted')  # label x
plt.ylabel('True')  # label y
plt.show()  # tampilkan plot


In [ ]:
# 9. Melakukan inferensi data train
# ambil beberapa sample dari data train dan prediksi
sample_idx = np.random.choice(X_train_flat.shape[0], size=10, replace=False)  # pilih sampel acak
sample_X = X_train_flat[sample_idx]  # ambil fitur sampel
sample_y = y_train[sample_idx]  # ambil label asli
sample_pred = np.argmax(model.predict(sample_X), axis=1)  # prediksi label sampel
print('True labels :', sample_y)  # tampilkan label asli
print('Predicted    :', sample_pred)  # tampilkan label prediksi


In [ ]:
# 10. Menampilkan data yang salah diprediksi
# cari indeks sample yang salah pada test set
mis_idx = np.where(y_pred != y_test)[0]  # indeks salah prediksi
if len(mis_idx) == 0:  # jika tidak ada salah prediksi
    print('Tidak ada sample yang salah diprediksi')  # informasikan
else:
    # tampilkan 9 sample salah prediksi jika tersedia
    display_idx = mis_idx[:9]  # ambil maksimal 9
    plt.figure(figsize=(8,8))  # ukuran plot
    for i, idx in enumerate(display_idx):  # iterasi tiap salah prediksi
        plt.subplot(3,3,i+1)  # subplot grid
        plt.imshow(X_test[idx], cmap='gray')  # tampilkan gambar asli
        plt.title(f'True:{y_test[idx]} Pred:{y_pred[idx]}')  # judul dengan label
        plt.axis('off')  # sembunyikan axis
    plt.tight_layout()  # rapikan layout
    plt.show()  # tampilkan


**Tugas Praktikum 5 — Fashion MNIST**
Kerjakan semua proses yang sama dengan praktikum sebelumnya menggunakan dataset Fashion MNIST: load data, tampilkan distribusi & sample, pra-proses, bangun dan latih MLP, tampilkan history, classification report, confusion matrix, inferensi, dan tampilkan contoh yang salah diprediksi.

In [ ]:
# 1. Load Dataset Fashion MNIST
from tensorflow.keras.datasets import fashion_mnist  # import fashion mnist
(Xf_train, yf_train), (Xf_test, yf_test) = fashion_mnist.load_data()  # load data
print('Shapes:', Xf_train.shape, yf_train.shape, Xf_test.shape, yf_test.shape)  # tampilkan shape

In [ ]:
# 2. Menampilkan distribusi data Test (Fashion MNIST)
import pandas as pd  # import pandas
pd.Series(yf_test).value_counts().sort_index().plot(kind='bar')  # plot distribusi label test
plt.title('Distribusi label - Test set (Fashion MNIST)')  # judul plot
plt.xlabel('Label')  # label x
plt.ylabel('Count')  # label y
plt.show()  # tampilkan plot

In [ ]:
# 3. Menampilkan sample data Train (Fashion MNIST)
plt.figure(figsize=(8,8))  # atur ukuran figura
for i in range(9):  # tampilkan 9 sample
    plt.subplot(3,3,i+1)  # grid 3x3
    plt.imshow(Xf_train[i], cmap='gray')  # tampilkan gambar grayscale
    plt.title(f'True: {yf_train[i]}')  # judul menampilkan label asli
    plt.axis('off')  # sembunyikan axis
plt.tight_layout()  # rapikan layout
plt.show()  # tampilkan semua gambar

In [ ]:
# 4. Pra-proses data (Fashion MNIST)
# normalisasi, flatten, dan one-hot encode label
Xf_train = Xf_train.astype('float32') / 255.0  # normalisasi ke [0,1]
Xf_test = Xf_test.astype('float32') / 255.0  # normalisasi test
ns, nx, ny = Xf_train.shape  # ukuran gambar
Xf_train_flat = Xf_train.reshape((ns, nx*ny))  # flatten
Xf_test_flat = Xf_test.reshape((Xf_test.shape[0], nx*ny))  # flatten test
yf_train_cat = to_categorical(yf_train, num_classes=10)  # one-hot encode train
yf_test_cat = to_categorical(yf_test, num_classes=10)  # one-hot encode test
print('After preprocessing shapes (Fashion):', Xf_train_flat.shape, yf_train_cat.shape, Xf_test_flat.shape, yf_test_cat.shape)

In [ ]:
# 5. Membuat model dan train model (Fashion MNIST)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
model_f = Sequential([
    Dense(512, activation='relu', input_shape=(nx*ny,)),  # hidden layer 1
    Dropout(0.2),  # dropout
    Dense(256, activation='relu'),  # hidden layer 2
    Dropout(0.2),  # dropout
    Dense(10, activation='softmax')  # output
])
model_f.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])  # compile
history_f = model_f.fit(Xf_train_flat, yf_train_cat, epochs=10, batch_size=128, validation_split=0.1)  # latih model

In [ ]:
# 6. Menampilkan grafik history train (Fashion MNIST)
plt.figure(figsize=(12,4))  # figure lebar
plt.subplot(1,2,1)  # subplot untuk loss
plt.plot(history_f.history['loss'], label='train loss')  # plot train loss
plt.plot(history_f.history['val_loss'], label='val loss')  # plot val loss
plt.title('Loss selama training (Fashion)')  # judul
plt.xlabel('Epoch')  # label x
plt.ylabel('Loss')  # label y
plt.legend()  # legend
plt.subplot(1,2,2)  # subplot untuk akurasi
plt.plot(history_f.history['accuracy'], label='train acc')  # plot train acc
plt.plot(history_f.history['val_accuracy'], label='val acc')  # plot val acc
plt.title('Akurasi selama training (Fashion)')  # judul
plt.xlabel('Epoch')  # label x
plt.ylabel('Accuracy')  # label y
plt.legend()  # legend
plt.show()  # tampilkan grafik

In [ ]:
# 7. Menampilkan classification report (Fashion MNIST)
yf_pred_proba = model_f.predict(Xf_test_flat)  # prediksi probabilitas pada test
yf_pred = np.argmax(yf_pred_proba, axis=1)  # ambil label tertinggi
from sklearn.metrics import classification_report, accuracy_score  # import metrics
print('Test accuracy (Fashion):', accuracy_score(yf_test, yf_pred))  # cetak akurasi
print('\nClassification Report (Fashion):\n', classification_report(yf_test, yf_pred))  # tampilkan report

In [ ]:
# 8. Membuat confusion matrix (Fashion MNIST)
from sklearn.metrics import confusion_matrix  # import confusion matrix
cm_f = confusion_matrix(yf_test, yf_pred)  # hitung confusion matrix
plt.figure(figsize=(10,8))  # set figure size
sns.heatmap(cm_f, annot=True, fmt='d', cmap='Blues')  # tampilkan heatmap
plt.title('Confusion Matrix (Fashion)')  # judul
plt.xlabel('Predicted')  # label x
plt.ylabel('True')  # label y
plt.show()  # tampilkan

In [ ]:
# 9. Melakukan inferensi data train (Fashion MNIST)
sample_idx_f = np.random.choice(Xf_train_flat.shape[0], size=10, replace=False)  # pilih sampel acak
sample_Xf = Xf_train_flat[sample_idx_f]  # ambil fitur sampel
sample_yf = yf_train[sample_idx_f]  # ambil label asli
sample_pred_f = np.argmax(model_f.predict(sample_Xf), axis=1)  # prediksi label sampel
print('True labels :', sample_yf)  # tampilkan label asli
print('Predicted    :', sample_pred_f)  # tampilkan label prediksi

In [ ]:
# 10. Menampilkan data yang salah diprediksi (Fashion MNIST)
mis_idx_f = np.where(yf_pred != yf_test)[0]  # indeks salah prediksi
if len(mis_idx_f) == 0:  # jika tidak ada salah prediksi
    print('Tidak ada sample yang salah diprediksi (Fashion)')  # informasikan
else:
    display_idx_f = mis_idx_f[:9]  # ambil maksimal 9
    plt.figure(figsize=(8,8))  # ukuran plot
    for i, idx in enumerate(display_idx_f):  # iterasi tiap salah prediksi
        plt.subplot(3,3,i+1)  # subplot grid
        plt.imshow(Xf_test[idx], cmap='gray')  # tampilkan gambar asli
        plt.title(f'True:{yf_test[idx]} Pred:{yf_pred[idx]}')  # judul dengan label
        plt.axis('off')  # sembunyikan axis
    plt.tight_layout()  # rapikan layout
    plt.show()  # tampilkan

**Instruksi Pembuatan Presentasi & Laporan**
- Buat presentasi (mis. `Presentasi_Praktikum5.pptx`) yang berisi: tujuan, arsitektur model, parameter training, grafik loss/accuracy, confusion matrix, contoh salah prediksi, dan kesimpulan.
- Buat laporan (mis. `Laporan_Praktikum5.md` atau PDF) yang memuat: ringkasan eksperimen, tabel metrik, analisis kesalahan, dan rekomendasi eksperimen lanjutan.

Jika Anda mau, saya bisa membuat template `Laporan_Praktikum5.md` dan outline presentasi secara otomatis. Mau saya buat sekarang?